# EDA Basis 1st Differences

# Setup

## Imports

In [ ]:
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss

from thesis_project import (
    config,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "eda-basis-diff"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

## Utilities

### Plotting

# Main

## Common

In [ ]:
# Basis data
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
basis_df = pd.read_parquet(path / filename)
basis_df = basis_df.astype("float32")
if not basis_df.index.is_monotonic_increasing:
    basis_df = basis_df.sort_index()
assert isinstance(basis_df.index, pd.DatetimeIndex)

idx = basis_df.index
tz = idx.tz

# BASIS_RESAMPLE_FREQS = [
#     "1s",
#     "5s",
#     "10s",
#     "15s",
#     "30s",
#     "1min",
# ]

BASIS_DELTAS = {
    # "1s": 1,  # seconds
    # "5s": 5,
    # "10s": 10,
    # "15s": 15,
    # "30s": 30,
    "1min": 60,
    # "5min": 300,
    "10min": 600,
    # "15min": 900,
    # "30min": 1800,
    # "1h": 3600,
}

# Compute basis 1st differences for different delta time
for key, delta in BASIS_DELTAS.items():
    basis_df[f"Basis Delta {key}"] = (
        basis_df["Basis"].groupby(basis_df.index.normalize()).diff(delta).astype("float32")
    )

In [ ]:
fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

In [ ]:
# Get filtered relevant dates
start = basis_df.index.min().normalize()
end = basis_df.index.max().normalize()

# Last trading dates of futures
FUT_LAST_TRADING_DATES = pd.to_datetime(fut_meta_df["Last Trading Date"].values)
FUT_LAST_TRADING_DATES = FUT_LAST_TRADING_DATES.tz_localize(tz).normalize()
FUT_LAST_TRADING_DATE_FILTERED = FUT_LAST_TRADING_DATES[
    (FUT_LAST_TRADING_DATES >= start) & (FUT_LAST_TRADING_DATES <= end)
]

# CTD switch dates
CTD_SWITCH_DATES = pd.to_datetime(ctd_switch_df["CTD Switch Date"].values)
CTD_SWITCH_DATES = CTD_SWITCH_DATES.tz_localize(tz).normalize()
CTD_SWITCH_DATES_FILTERED = CTD_SWITCH_DATES[
    (CTD_SWITCH_DATES >= start) & (CTD_SWITCH_DATES <= end)
]

## Distribution

In [ ]:
info = []
for key, delta in BASIS_DELTAS.items():
    basis_delta = basis_df[f"Basis Delta {key}"]
    stats = basis_delta.describe()
    stats["skewness"] = basis_delta.skew()
    stats["kurtosis"] = basis_delta.kurtosis()
    stats.name = f"{key}"
    info.append(stats)
info = pd.DataFrame(info)
info.index.name = "Delta"

# info.to_csv(OUT_TAB_DIR / "gross_basis_diff_describe_summary.csv")

display(info)

In [ ]:
import matplotlib.ticker as mtick

# Dates
start_date = basis_df.index.min()
end_date = basis_df.index.max()

subset = BASIS_DELTAS.keys()  # Plot all deltas
filtered = {key: BASIS_DELTAS[key] for key in subset}

n_plots = len(filtered)
cols = min(2, n_plots)
rows = (n_plots + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows), constrained_layout=True)
axes = axes.flatten()

for ax, (key, delta) in zip(axes, filtered.items()):
    # Cut tails
    plot_data = basis_df[f"Basis Delta {key}"]
    upper = plot_data.quantile(0.999)
    lower = plot_data.quantile(0.001)
    plot_data = plot_data[plot_data.between(lower, upper)]

    # Weights
    weights = np.ones_like(plot_data) / len(plot_data)

    # Plot
    ax.hist(
        plot_data,
        bins=750,
        weights=weights,
    )

    # Stats
    median = plot_data.median()
    mean = plot_data.mean()
    std = plot_data.std()

    # Lines
    ax.axvline(
        median, color="green", linestyle="--", linewidth=0.75, label=f"Median ({median:.4f})"
    )
    ax.axvline(mean, color="red", linestyle="--", linewidth=0.75, label=f"Mean ({mean:.4f})")
    ax.axvline(mean + std, color="orange", linestyle="--", linewidth=0.75, label=r"$\pm$ 1 Std")
    ax.axvline(mean - std, color="orange", linestyle="--", linewidth=0.75)

    # Labels
    ax.set_title(f"Delta {key}")
    ax.set_xlabel("Basis 1st Difference")
    ax.set_ylabel("Frequency")
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(loc="upper right")

# Suptitle
fig.suptitle(
    f"Basis 1st Difference Distributions"
    f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    f"\nQuantiles 0.1% - 99.9%",
    fontsize=16,
)

for ax in axes[n_plots:]:
    ax.remove()

# fig.savefig(OUT_FIG_DIR / "basis_diff_distribution_sample.png", dpi=300)

plt.show()

## Timeseries

In [ ]:
# Plot basis 1st difference time series
import pandas as pd

# Dates
start_date = basis_df.index.min()
end_date = basis_df.index.max()

# Plots limits
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

subset = [
    "1s",
    "5s",
    "10s",
    "15s",
    "30s",
]
filtered = {key: BASIS_DELTAS[key] for key in subset}

# Grid
n_plots = len(filtered)
cols = min(2, n_plots)
rows = (n_plots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows), constrained_layout=True)
axes = axes.flatten()

for ax, (key, freq) in zip(axes, filtered.items()):
    # Downsample
    plot_data = basis_df[f"Basis Delta {key}"].resample("1min").last().dropna()

    # Plot
    ax.plot(plot_data, linewidth=0.5, label=f"{key}")

    # Labels
    ax.set_title(f"Delta {key}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Basis 1st Difference")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(start_xlim, end_xlim)

    # X ticks
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3, tz=tz))

    # Vertical lines
    for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        ax.axvline(
            d,
            color="black",
            linestyle="--",
            linewidth=0.5,
            label="Last Trading Date" if i == 0 else None,
        )
    # for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
    #     ax.axvline(
    #         d,
    #         color="blue",
    #         linestyle="--",
    #         linewidth=0.5,
    #         label="CTD Switch Date" if i == 0 else None
    #     )

    ax.legend(loc="upper right")

# Remove unused axes
for ax in axes[n_plots:]:
    ax.remove()

# Suptitle
fig.suptitle(
    f"Basis 1st Difference Time Series\n"
    f"From {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}",
    fontsize=16,
)

fig.savefig(OUT_FIG_DIR / "basis_diff_timeseries_sample.png", dpi=300)

plt.show()

### Day Sample

In [ ]:
import matplotlib.dates as mdates
import pandas as pd

# Date setup
assert isinstance(basis_df.index, pd.DatetimeIndex)
year, month, day = "2022", "09", "07"
date = pd.Timestamp(f"{year}-{month}-{day}", tz=basis_df.index.tz)
date_mask = basis_df.index.normalize() == date

start_xlim = date + pd.Timedelta(hours=7, minutes=30)
end_xlim = date + pd.Timedelta(hours=18, minutes=30)

# Grid
subset = [
    "5s",
    "30s",
    "10min",
]
filtered = {key: BASIS_DELTAS[key] for key in subset}

n_plots = len(filtered)
cols = min(3, n_plots)
rows = (n_plots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 4 * rows), constrained_layout=True)
axes = axes.flatten()

for ax, (key, freq) in zip(axes, filtered.items()):
    plot_data = basis_df.loc[date_mask, f"Basis Delta {key}"]

    ax.plot(plot_data, linewidth=0.5, label=f"Delta {key}")

    # Labels
    ax.set_title(f"Delta {key}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Basis 1st Difference")
    ax.grid(True, alpha=0.3)

    # Time axis formatting
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=basis_df.index.tz))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=basis_df.index.tz))

    ax.set_xlim(start_xlim, end_xlim)

# Remove unused axes
for ax in axes[n_plots:]:
    ax.remove()

# Suptitle
fig.suptitle(f"Basis 1st Difference Intraday Profiles\n{date.date()}")

plt.show()

## Rolling Statistics

In [ ]:
ROLLING_STAT_WINDOWS = {
    "10s": 10,
    "30s": 30,
    "1min": 60,
    "5min": 300,
    "10min": 600,
    "30min": 1800,
    "60min": 3600,
}

ROLLING_STATS = [
    "mean",
    "std",
]

In [ ]:
# Compute rolling statistics for basis 1st difference
assert isinstance(basis_df.index, pd.DatetimeIndex)

rolling_stats_df = pd.DataFrame(index=basis_df.index)

day = basis_df.index.date
cols = []
for key_d in BASIS_DELTAS:
    s = basis_df[f"Basis Delta {key_d}"]
    grouped = s.groupby(day)

    for key_w, window in ROLLING_STAT_WINDOWS.items():
        w = f"{window}s"

        mean = grouped.transform(lambda x: x.rolling(w).mean().astype("float32"))
        std = grouped.transform(lambda x: x.rolling(w).std().astype("float32"))

        mean.name = f"Basis Delta {key_d} Rolling Mean {key_w}"
        std.name = f"Basis Delta {key_d} Rolling Std {key_w}"

        cols.append(mean)
        cols.append(std)

rolling_stats_df = pd.concat(cols, axis=1)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_diff_rolling_stats.parquet"
# rolling_stats_df.to_parquet(path / filename)
rolling_stats_df = pd.read_parquet(path / filename)

In [ ]:
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)
resample_freq = "1min"

subset = ROLLING_STAT_WINDOWS.keys()
filtered = {key: ROLLING_STAT_WINDOWS[key] for key in subset}

n_plots = len(filtered)
cols = min(3, n_plots)
rows = (n_plots + cols - 1) // cols

for key_d, delta in BASIS_DELTAS.items():
    fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows))
    axes = axes.flatten()

    for ax, (key_w, window) in zip(axes, filtered.items()):
        w = f"{window}s"
        for stat in ROLLING_STATS:
            plot_data = (
                rolling_stats_df[f"Basis Delta {key_d} Rolling {stat} {key_w}"]
                .resample(resample_freq)
                .last()
                .dropna()
            )
            ax.plot(
                plot_data,
                label=f"Rolling {stat}",
                linewidth=0.5,
            )
        title = f"Rolling window {key_w}"
        ax.set_title(title)
        ax.set_xlabel("Time")
        ax.set_ylabel("Rolling Statistics")
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
        ax.set_xlim(start_xlim, end_xlim)

        for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
            ax.axvline(
                d,
                color="red",
                linestyle="--",
                linewidth=0.5,
                label="Last Trading Date" if i == 0 else None,
            )
        # for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
        #     ax.axvline(
        #         d,
        #         color="blue",
        #         linestyle="--",
        #         linewidth=0.5,
        #         label="CTD Switch Date" if i == 0 else None
        #     )
        ax.legend(loc="upper right")

    suptitle = (
        f"Basis 1st Difference Rolling Statistics (Delta {key_d})\n"
        f"From {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    )
    fig.suptitle(suptitle, fontsize=16)

    for ax in axes[n_plots:]:
        ax.remove()

    plt.show()

### Day sample

In [ ]:
# Plot a sample day of basis 1st differences rolling statistics
assert isinstance(rolling_stats_df.index, pd.DatetimeIndex)
year, month, day = "2022", "09", "07"
date = pd.Timestamp(f"{year}-{month}-{day}", tz=rolling_stats_df.index.tz)

# Filter plot data
date_mask = rolling_stats_df.index.normalize() == date
plot_data = rolling_stats_df[date_mask]

start_xlim = date + pd.Timedelta(hours=7, minutes=30)
end_xlim = date + pd.Timedelta(hours=18, minutes=30)

subset = ROLLING_STAT_WINDOWS.keys()
filtered = {key: ROLLING_STAT_WINDOWS[key] for key in subset}

n_plots = len(filtered)
cols = min(3, n_plots)
rows = (n_plots + cols - 1) // cols

for key_d, delta in BASIS_DELTAS.items():
    fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows))
    axes = axes.flatten()

    for ax, (key_w, window) in zip(axes, filtered.items()):
        for stat in ROLLING_STATS:
            ax.plot(
                plot_data[f"Basis Delta {key_d} Rolling {stat.capitalize()} {key_w}"],
                label=f"Rolling {stat}",
                linewidth=0.5,
            )

        title = f"Rollling window {key}"
        ax.set_title(title)
        ax.set_xlabel("Time")
        ax.set_ylabel("Rolling Statistics")
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=tz))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=tz))
        ax.set_xlim(start_xlim, end_xlim)

        axt = ax
        axt.plot(
            basis_df.loc[date_mask, "Basis"],
            label="Basis",
            color="gray",
            alpha=0.5,
            linewidth=0.5,
        )
        # axt.set_ylabel("Basis")
        # axt.grid(False)

        # handles1, labels1 = ax.get_legend_handles_labels()
        # handles2, labels2 = axt.get_legend_handles_labels()
        # ax.legend(handles1 + handles2, labels1 + labels2, loc="upper left")
        ax.legend(loc="upper left")

    for ax in axes[n_plots:]:
        ax.remove()

    suptitle = (
        f"Basis 1st Differences (Delta {key_d}) Rolling Statistics on {date.strftime('%Y-%m-%d')}"
    )
    fig.suptitle(suptitle, fontsize=16)
    plt.show()

## ACF and PACF

### Basis 1st Differences

In [ ]:
nlags = 300
lags = list(range(nlags + 1))
alpha = 0.05

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

assert isinstance(basis_df.index, pd.DatetimeIndex)

subset = BASIS_DELTAS.keys()
filtered = {key: BASIS_DELTAS[key] for key in subset}

dates = basis_df.index.normalize()
groups_idx = {d: np.where(dates == d)[0] for d in np.unique(dates)}

all_acf = []
all_acf_cols = []
all_pacf = []
all_pacf_cols = []
index = None

for key_d in filtered:
    s = basis_df[f"Basis Delta {key_d}"].values.astype("float32")

    acf_vals_list = []
    acf_low_list = []
    acf_up_list = []

    pacf_vals_list = []
    pacf_low_list = []
    pacf_up_list = []

    idx_list = []

    acf_cols = (
        [f"Delta {key_d} Lag {i}" for i in lags]
        + [f"Delta {key_d} Low CI {i}" for i in lags]
        + [f"Delta {key_d} Up CI {i}" for i in lags]
    )

    pacf_cols = (
        [f"Delta {key_d} Lag {i}" for i in lags]
        + [f"Delta {key_d} Low CI {i}" for i in lags]
        + [f"Delta {key_d} Up CI {i}" for i in lags]
    )

    all_acf_cols.extend(acf_cols)
    all_pacf_cols.extend(pacf_cols)

    for d, idx in groups_idx.items():
        x = s[idx]
        x = x[~np.isnan(x)]
        if len(x) < nlags + 1:
            continue

        acf_vals, acf_conf = acf(x, nlags=nlags, alpha=alpha, fft=True)  # type: ignore
        pacf_vals, pacf_conf = pacf(x, nlags=nlags, alpha=alpha, method="burg")

        acf_vals_list.append(acf_vals)
        acf_low_list.append(acf_conf[:, 0])
        acf_up_list.append(acf_conf[:, 1])

        pacf_vals_list.append(pacf_vals)
        pacf_low_list.append(pacf_conf[:, 0])
        pacf_up_list.append(pacf_conf[:, 1])

        idx_list.append(d)

    if index is None:
        index = pd.Index(idx_list)

    acf_block = np.hstack(
        [
            np.vstack(acf_vals_list),
            np.vstack(acf_low_list),
            np.vstack(acf_up_list),
        ]
    )

    pacf_block = np.hstack(
        [
            np.vstack(pacf_vals_list),
            np.vstack(pacf_low_list),
            np.vstack(pacf_up_list),
        ]
    )

    all_acf.append(acf_block)
    all_pacf.append(pacf_block)

delta_acf_df = pd.DataFrame(
    np.hstack(all_acf),
    index=index,
    columns=all_acf_cols,
)

delta_pacf_df = pd.DataFrame(
    np.hstack(all_pacf),
    index=index,
    columns=all_pacf_cols,
)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"

filename = "gross_basis_diff_acf.parquet"
# delta_acf_df.to_parquet(path / filename)
delta_acf_df = pd.read_parquet(path / filename)
display(delta_acf_df.head())

filename = "gross_basis_diff_pacf.parquet"
# delta_pacf_df.to_parquet(path / filename)
delta_pacf_df = pd.read_parquet(path / filename)
display(delta_pacf_df.head())

#### Per Lag Time Series

In [ ]:
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

lag_subset = range(1, 2)

subset = [
    "30s",
    "5min",
    "10min",
    "1h",
]
filtered = {key: BASIS_DELTAS[key] for key in subset}

n_plots = len(filtered)
cols = min(2, n_plots)
rows = (n_plots + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows), constrained_layout=True)
axes = axes.flatten()

x = delta_acf_df.index

for ax, key_d in zip(axes, filtered.keys()):
    for lag in lag_subset:
        acf_vals = delta_acf_df[f"Delta {key_d} Lag {lag}"]
        acf_low = delta_acf_df[f"Delta {key_d} Low CI {lag}"]
        acf_up = delta_acf_df[f"Delta {key_d} Up CI {lag}"]
        ax.plot(x, acf_vals, linewidth=0.5, label=f"ACF Lag {lag}")
        ax.fill_between(x, acf_low, acf_up, alpha=0.2)

        pacf_vals = delta_pacf_df[f"Delta {key_d} Lag {lag}"]
        pacf_low = delta_pacf_df[f"Delta {key_d} Low CI {lag}"]
        pacf_up = delta_pacf_df[f"Delta {key_d} Up CI {lag}"]
        ax.plot(x, pacf_vals, linewidth=0.5, linestyle="-", label=f"PACF Lag {lag}")
        ax.fill_between(x, pacf_low, pacf_up, alpha=0.2)

    ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(range(1, 13), interval=1, tz=tz))
    ax.set_xlim(start_xlim, end_xlim)

    for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        ax.axvline(
            d,
            color="black",
            linestyle="--",
            linewidth=0.5,
            label="Futures Last Trading Date" if i == 0 else None,
        )

    # for i, d in enumerate(CTD_SWITCH_DATES_FILTERED):
    #     ax.axvline(
    #         d,
    #         color='green',
    #         linestyle='--',
    #         linewidth=0.5,
    #         label='CTD Switch Date' if i == 0 else None
    #     )

    ax.axhline(0, color="red", linestyle="-", linewidth=0.5)

    title = (
        f"Delta {key_d}"
        f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
        f"\nPlotted Lags: ({', '.join(str(lag) for lag in subset)})"
    )
    ax.set_title(title)
    ax.set_xlabel("Day")
    ax.set_ylabel("Correlation")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

for ax in axes[n_plots:]:
    ax.remove()

suptitle = (
    f"Basis 1st Differences ACF and PACF\n"
    f"From {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    f"\nPlotted Lags: ({', '.join(str(lag) for lag in subset)})"
)
fig.suptitle(suptitle, fontsize=16)

plt.show()

#### Average Lags Time Series

In [ ]:
ticks = [f"Lag {i}" for i in range(0, nlags + 1)]

subset = [
    "30s",
    "5min",
    "10min",
    "1h",
]
filtered = {key: BASIS_DELTAS[key] for key in subset}

n_plots = len(filtered)
cols = min(2, n_plots)
rows = (n_plots + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12 * cols, 6 * rows))
axes = axes.flatten()

for ax, key_d in zip(axes, filtered.keys()):
    plot_data = delta_acf_df.mean()[[f"Delta {key_d} Lag {i}" for i in range(nlags + 1)]]
    low = delta_acf_df.mean()[[f"Delta {key_d} Low CI {i}" for i in range(nlags + 1)]]
    up = delta_acf_df.mean()[[f"Delta {key_d} Up CI {i}" for i in range(nlags + 1)]]
    ax.plot(
        ticks,
        plot_data,
        label="ACF",
        marker="o",
        markersize=4,
        linestyle="-",
        linewidth=0.5,
        alpha=0.7,
    )
    ax.fill_between(ticks, low, up, alpha=0.2, label="ACF Confidence Interval")

    plot_data = delta_pacf_df.mean(axis=0)[[f"Delta {key_d} Lag {i}" for i in range(nlags + 1)]]
    low = delta_pacf_df.mean(axis=0)[[f"Delta {key_d} Low CI {i}" for i in range(nlags + 1)]]
    up = delta_pacf_df.mean(axis=0)[[f"Delta {key_d} Up CI {i}" for i in range(nlags + 1)]]
    ax.plot(
        ticks,
        plot_data,
        label="PACF",
        marker="o",
        markersize=4,
        linestyle="-",
        linewidth=0.5,
        alpha=0.7,
    )
    ax.fill_between(ticks, low, up, alpha=0.2, label="PACF Confidence Interval")

    ax.axhline(0, color="red", linestyle="-", linewidth=0.5)

    ax.set_title(f"Delta {key_d}")
    ax.set_xlabel("Lag")
    ax.set_ylabel("Correlation")
    ax.grid(True, alpha=0.3)
    ax.set_xticks(ticks[::15])
    ax.set_xticks(ticks[::5], minor=True)
    ax.legend()

for ax in axes[n_plots:]:
    ax.remove()

suptitle = "Average ACF and PACF of Basis 1st Differences\n"
fig.suptitle(suptitle, fontsize=16)

# fig.savefig(OUT_FIG_DIR / "basis_diff_acf_pacf_sample.png", dpi=300)

plt.tight_layout()
plt.show()

## Mean Reversion

### Day-wise Stationarity

In [ ]:
def check_stationarity(series, regression):
    adf_stat, adf_pvalue, *_ = adfuller(series, regression=regression)
    kpss_stat, kpss_pvalue, *_ = kpss(series, regression=regression)

    return {
        "ADF": {
            "Statistic": adf_stat,
            "p-value": adf_pvalue,
        },
        "KPSS": {
            "Statistic": kpss_stat,
            "p-value": kpss_pvalue,
        },
    }


STATIONARITY_TESTS = {
    "c": "Constant",
    "ct": "Constant and Trend",
}

In [ ]:
# Perform day-wise stationarity tests on 1st differences
assert isinstance(basis_df.index, pd.DatetimeIndex)
warnings.filterwarnings("ignore", category=InterpolationWarning)

results = []

for date, group in basis_df.groupby(basis_df.index.normalize()):
    row = {"Date": date.date()}

    for key, delta in BASIS_DELTAS.items():
        delta = group[f"Basis Delta {key}"].dropna()

        for reg, reg_name in STATIONARITY_TESTS.items():
            out = check_stationarity(delta, reg)
            row.update(
                {
                    f"ADF Statistic Delta {key} {reg_name}": out["ADF"]["Statistic"],
                    f"ADF p-value Delta {key} {reg_name}": out["ADF"]["p-value"],
                    f"KPSS Statistic Delta {key} {reg_name}": out["KPSS"]["Statistic"],
                    f"KPSS p-value Delta {key} {reg_name}": out["KPSS"]["p-value"],
                }
            )

    results.append(row)

results_df = pd.DataFrame.from_records(results).set_index("Date")

In [ ]:
# Perform day-wise stationarity tests on 1st differences
assert isinstance(basis_df.index, pd.DatetimeIndex)
warnings.filterwarnings("ignore", category=InterpolationWarning)

cols = {key: basis_df[f"Basis Delta {key}"].values.astype("float32") for key in BASIS_DELTAS}

dates = basis_df.index.normalize()
unique_dates = np.unique(dates)
groups_idx = {d: np.where(dates == d)[0] for d in unique_dates}

columns = []
for key in BASIS_DELTAS:
    for reg, reg_name in STATIONARITY_TESTS.items():
        columns.extend(
            [
                f"ADF Statistic Delta {key} {reg_name}",
                f"ADF p-value Delta {key} {reg_name}",
                f"KPSS Statistic Delta {key} {reg_name}",
                f"KPSS p-value Delta {key} {reg_name}",
            ]
        )

n_rows = len(unique_dates)
n_cols = len(columns)
data = np.full((n_rows, n_cols), np.nan, dtype="float32")

for i, d in enumerate(unique_dates):
    idx = groups_idx[d]
    col_idx = 0

    for key in BASIS_DELTAS:
        x = cols[key][idx]
        x = x[~np.isnan(x)]

        for reg, reg_name in STATIONARITY_TESTS.items():
            out = check_stationarity(x, reg)

            data[i, col_idx] = out["ADF"]["Statistic"]
            data[i, col_idx + 1] = out["ADF"]["p-value"]
            data[i, col_idx + 2] = out["KPSS"]["Statistic"]
            data[i, col_idx + 3] = out["KPSS"]["p-value"]

            col_idx += 4

results_df = pd.DataFrame(
    data, index=pd.Index([d.date() for d in unique_dates], name="Date"), columns=columns
)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_diff_stationarity_tests.parquet"
# results_df.to_parquet(path / filename)
results_df = pd.read_parquet(path / filename)
display(results_df.head())

#### Summary

In [ ]:
import re


def to_seconds(s):
    match = re.match(r"(\d+)([a-zA-Z]+)", s)
    value, unit = match.groups()  # type: ignore
    value = int(value)

    multipliers = {
        "s": 1,
        "min": 60,
        "h": 3600,
    }

    return value * multipliers[unit]


summary = []

for reg, reg_name in STATIONARITY_TESTS.items():
    for key, delta in BASIS_DELTAS.items():
        adf_col = f"ADF p-value Delta {key} {reg_name}"
        kpss_col = f"KPSS p-value Delta {key} {reg_name}"
        adf_ratio = (results_df[adf_col] <= 0.05).mean()
        kpss_ratio = (results_df[kpss_col] > 0.05).mean()
        summary.append(
            {
                "Regression": reg_name,
                "Delta": key,
                "ADF Stationary Ratio": f"{adf_ratio * 100:.2f} %",
                "KPSS Stationary Ratio": f"{kpss_ratio * 100:.2f} %",
            }
        )

summary = (
    pd.DataFrame(summary)
    .sort_values(
        by=["Regression", "Delta"],
        key=lambda col: col.map(lambda x: to_seconds(x) if col.name == "Delta" else x),
    )
    .set_index(["Regression", "Delta"])
)

# summary.to_csv(OUT_TAB_DIR / "gross_basis_diff_stationarity_summary.csv")

display(summary)

#### p-value Distribution

In [ ]:
import numpy as np
from matplotlib import ticker as mtick

bins = np.linspace(0, 1, 101)

for reg, reg_name in STATIONARITY_TESTS.items():
    n_plots = len(BASIS_DELTAS)
    cols = min(4, n_plots)
    rows = (n_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 8), constrained_layout=True)
    axes = axes.flatten()

    for (key, delta), ax in zip(BASIS_DELTAS.items(), axes):
        for test in ["ADF", "KPSS"]:
            plot_data = results_df[f"{test} p-value Delta {key} {reg_name}"]
            weights = np.ones_like(plot_data) / len(plot_data)
            ax.hist(
                plot_data,
                bins=bins,  # type: ignore
                alpha=0.5,
                label=f"{test}",
                weights=weights,
            )

        ax.set_title(f"Delta {key}")
        ax.set_xlabel("p-value")
        ax.set_ylabel("Frequency")
        ax.set_xticks(bins[::5])
        ax.set_xticks(bins[::1], minor=True)
        ax.set_xlim(0, 0.25)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.axvline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

    for ax in axes[n_plots:]:
        ax.remove()

    suptitle = (
        f"Basis 1st Differences Daily Stationarity Tests"
        f"\np-value Distributions"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    # fig.savefig(OUT_FIG_DIR / f"basis_diff_stationarity_pvalue_distribution_{reg_name.replace(' ', '_').lower()}.png", dpi=300)

    plt.show()

#### p-value Time Series

In [ ]:
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

n_plots = len(BASIS_DELTAS)
cols = min(3, n_plots)
rows = (n_plots + cols - 1) // cols

for reg, reg_name in STATIONARITY_TESTS.items():
    fig, axes = plt.subplots(rows, cols, figsize=(16, 12), constrained_layout=True)
    axes = axes.flatten()

    for ax, (key, delta) in zip(axes, BASIS_DELTAS.items()):
        for test in ["ADF", "KPSS"]:
            plot_data = results_df[f"{test} p-value Delta {key} {reg_name}"]
            ax.plot(
                plot_data,
                label=f"{test} p-value",
                linewidth=0.5,
                alpha=0.7,
            )

        ax.axhline(0.05, color="red", linestyle="-", linewidth=0.5, label="Significance Level (5%)")
        # for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        #     ax.axvline(
        #         d,
        #         color='black',
        #         linestyle='--',
        #         linewidth=0.5,
        #         label='Futures Last Trading Date' if i == 0 else None
        #     )
        title = f"Delta: {key}"
        ax.set_title(title)
        ax.set_xlabel("Date")
        ax.set_ylabel("p-value")
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=4, tz=tz))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(range(1, 13), interval=1, tz=tz))
        ax.set_xlim(start_xlim, end_xlim)

    for ax in axes[n_plots:]:
        ax.remove()

    suptitle = (
        f"Basis 1st Differences Daily Stationarity Tests"
        f"\np-value Time Series"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle, fontsize=16)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    # fig.savefig(
    #     OUT_FIG_DIR / f"basis_diff_stationarity_pvalue_timeseries_{reg_name.replace(' ', '_').lower()}.png",
    #     dpi=300,
    # )

    plt.show()

#### Statistic Time Series

In [ ]:
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

ADF_CRIT_VALUES = {
    "c": {
        "1%": -3.43054422,
        "5%": -2.86162584,
        "10%": -2.56681569,
    },
    "ct": {
        "1%": -3.95903889,
        "5%": -3.4106204,
        "10%": -3.12712679,
    },
}

KPSS_CRIT_VALUES = {
    "c": {
        "1%": 0.739,
        "5%": 0.463,
        "10%": 0.347,
    },
    "ct": {
        "1%": 0.216,
        "5%": 0.146,
        "10%": 0.119,
    },
}

n_plots = len(BASIS_DELTAS)
cols = 2
rows = (n_plots + cols - 1) // cols

for reg, reg_name in STATIONARITY_TESTS.items():
    fig, axes = plt.subplots(rows, cols, figsize=(16, 12))
    axes = axes.flatten()

    for ax, (key, delta) in zip(axes, BASIS_DELTAS.items()):
        plot_data = results_df[f"ADF Statistic Delta {key} {reg_name}"]
        ax.plot(
            plot_data,
            label="ADF Statistic",
            linewidth=0.5,
        )

        axt = ax.twinx()
        plot_data = results_df[f"KPSS Statistic Delta {key} {reg_name}"]
        axt.plot(
            plot_data,
            label="KPSS Statistic",
            color="orange",
            linewidth=0.5,
        )

        for p, crit in ADF_CRIT_VALUES[reg].items():
            if p == "5%":
                ax.axhline(
                    crit,
                    color="red",
                    linestyle="--",
                    linewidth=0.5,
                    label=f"ADF Critical Value {p}",
                )

        for p, crit in KPSS_CRIT_VALUES[reg].items():
            if p == "5%":
                axt.axhline(
                    crit,
                    color="blue",
                    linestyle="--",
                    linewidth=0.5,
                    label=f"KPSS Critical Value {p}",
                )

        # for i, d in enumerate(FUT_LAST_TRADING_DATE_FILTERED):
        #     ax.axvline(
        #         d,
        #         color='black',
        #         linestyle='--',
        #         linewidth=0.5,
        #         label='Futures Last Trading Date' if i == 0 else None
        #     )

        title = f"Delta: {key}"
        ax.set_title(title)
        ax.set_xlabel("Date")
        ax.set_ylabel("ADF Statistic")
        axt.set_ylabel("KPSS Statistic")
        ax.grid(True, alpha=0.3)
        axt.grid(False)
        ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=4, tz=tz))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m", tz=tz))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(range(1, 13), interval=1, tz=tz))
        ax.set_xlim(start_xlim, end_xlim)

    for ax in axes[n_plots:]:
        ax.remove()

    suptitle = (
        f"Basis 1st Differences Daily Stationarity Tests"
        f"\np-value Time Series"
        f"\nRegression Hypothesis: {reg_name}"
    )
    fig.suptitle(suptitle, fontsize=16)
    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")
    fig.tight_layout()
    plt.show()

### Hurst's Exponent

In [ ]:
# TODO: Implement hurst exponent check

## Boxplots

### Utilities

In [ ]:
import os
from typing import Optional


def plot_grouped_boxplot(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    order: list,
    tick_labels: list,
    showfliers: bool = False,
    title: Optional[str] = None,
    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    tick_positions: Optional[list] = None,
    save_path: Optional[str] = None,
):
    plot_data = [df[df[group_col] == val][value_col].dropna().values for val in order]

    fig, ax = plt.subplots(figsize=(6, 4))

    # tick_labels is only passed if tick_positions is None, otherwise we set it explicitly later
    ax.boxplot(
        plot_data,
        tick_labels=tick_labels if tick_positions is None else None,
        showfliers=showfliers,
    )

    if title:
        ax.set_title(title)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)

    if tick_positions is not None:
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([tick_labels[i - 1] for i in tick_positions])

    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.show()

In [ ]:
# Save path
save_path = None
SAVE = False

# Data copy for boxplot purposes
boxplot_df = basis_df.copy()
boxplot_df["weekday_num"] = boxplot_df.index.dayofweek

# Year
boxplot_df["year"] = boxplot_df.index.year

# Hour of day
boxplot_df["hour"] = boxplot_df.index.hour
hourly_order = list(range(boxplot_df["hour"].min(), boxplot_df["hour"].max() + 1))

# Day of week
boxplot_df["weekday"] = boxplot_df.index.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
boxplot_df["weekday"] = pd.Categorical(
    boxplot_df["weekday"], categories=weekday_order, ordered=True
)

# Week of year
boxplot_df["week_of_year"] = boxplot_df.index.isocalendar().week
weekly_order = sorted(boxplot_df["week_of_year"].unique())

# Month of year
boxplot_df["month"] = boxplot_df.index.month
boxplot_df["month_name"] = boxplot_df.index.strftime("%B")
monthly_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
boxplot_df["month_name"] = pd.Categorical(
    boxplot_df["month_name"], categories=monthly_order, ordered=True
)

### Hour of Day

In [ ]:
for key, delta in BASIS_DELTAS.items():
    title_base = (
        f"Basis 1st Difference Delta {key} Distribution by Dayhour\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_bp_by_hour_{'w' if showfliers else 'wo'}_outliers.png"
            )
            if SAVE
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis Delta {key}",
            group_col="hour",
            order=hourly_order,
            tick_labels=[f"{h:02d}:00" for h in hourly_order],
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis 1st Difference",
            xlabel="Hour of Day",
            save_path=save_path,
        )

### Day of Week

In [ ]:
for key, freq in BASIS_DELTAS.items():
    title_base = (
        f"Basis 1st Difference {key} Distribution by Weekday\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_by_weekday_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis diff {key}",
            group_col="weekday",
            order=weekday_order,
            tick_labels=weekday_order,
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis 1st Difference",
            xlabel="Day of Week",
            save_path=save_path,
        )

### Week of Year

In [ ]:
for key, freq in BASIS_DELTAS.items():
    title_base = (
        f"Basis 1st Difference {key} Distribution by Yearweek\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    tick_positions = [i for i, val in enumerate(weekly_order, start=1) if val % 4 == 0]
    tick_labels = [f"W{val:02d}" for val in weekly_order]

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_by_week_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis diff {key}",
            group_col="week_of_year",
            order=weekly_order,
            tick_labels=tick_labels,
            tick_positions=tick_positions,
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis Difference",
            xlabel="Week of Year",
            save_path=save_path,
        )

    # Split by year
    for year in boxplot_df["year"].unique():
        year_df = boxplot_df[boxplot_df["year"] == year]
        year_weekly_order = sorted(year_df["week_of_year"].unique())
        year_tick_positions = [
            i for i, val in enumerate(year_weekly_order, start=1) if val % 4 == 0
        ]
        year_tick_labels = [f"W{val:02d}" for val in year_weekly_order]

        for showfliers in [False, True]:
            save_path = (
                str(
                    OUT_FIG_DIR
                    / f"basis_diff_{key}_by_week_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
                )
                if NOTEBOOK is not None
                else None
            )
            plot_grouped_boxplot(
                df=year_df,
                value_col=f"Basis diff {key}",
                group_col="week_of_year",
                order=year_weekly_order,
                tick_labels=year_tick_labels,
                tick_positions=year_tick_positions,
                showfliers=showfliers,
                title=f"Basis 1st Diff {key} by Week of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
                ylabel="Basis Difference",
                xlabel="Week of Year",
                save_path=save_path,
            )

### Month of Year

In [ ]:
for key, freq in BASIS_DELTAS.items():
    title_base = (
        f"Basis 1st Difference {key} Distribution by Yearmonth\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_by_month_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis diff {key}",
            group_col="month_name",
            order=monthly_order,
            tick_labels=monthly_order,
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis 1st Difference",
            xlabel="Month of Year",
            save_path=save_path,
        )

    # Split by year
    for year in boxplot_df["year"].unique():
        year_df = boxplot_df[boxplot_df["year"] == year]

        for showfliers in [False, True]:
            save_path = (
                str(
                    OUT_FIG_DIR
                    / f"basis_diff_{key}_by_month_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
                )
                if NOTEBOOK is not None
                else None
            )
            plot_grouped_boxplot(
                df=year_df,
                value_col=f"Basis diff {key}",
                group_col="month_name",
                order=monthly_order,
                tick_labels=monthly_order,
                showfliers=showfliers,
                title=f"Basis 1st Diff {key} by Month of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
                ylabel="Basis 1st Difference",
                xlabel="Month of Year",
                save_path=save_path,
            )

### Contract Month

In [ ]:
# TODO